In [29]:
import random
import time
from typing import List, Dict, Set, Tuple, Optional
import numpy as np
import os
import asyncio
from azure.identity import DefaultAzureCredential
from langchain_openai import AzureChatOpenAI

# Define personality templates that will guide agent behavior
PERSONALITY_TEMPLATES = {
    "Suspicious": {
        "description": "Always suspicious of others, questions everything",
        "trust_threshold": 0.2,  # Low trust of others
        "aggression": 0.7,       # Likely to accuse others
        "deception": 0.4,        # Moderately good at lying
        "loyalty": 0.5,          # Average loyalty to team
        "prompt_style": "You are a very suspicious person who questions everything and trusts no one easily. You're quick to accuse others and constantly looking for inconsistencies. You often use phrases like 'I don't believe that', 'That sounds suspicious', and 'Why should we trust you?'"
    },
    "Trusting": {
        "description": "Tends to believe others, gives benefit of doubt",
        "trust_threshold": 0.8,  # High trust of others
        "aggression": 0.3,       # Less likely to accuse others
        "deception": 0.3,        # Not good at lying
        "loyalty": 0.7,          # High loyalty to team
        "prompt_style": "You are a very trusting person who tends to believe others and give them the benefit of the doubt. You're hesitant to accuse people without strong evidence and often defend others. You frequently use phrases like 'I believe you', 'That makes sense', and 'I trust what you're saying'."
    },
    "Strategic": {
        "description": "Calculates odds, makes logical deductions",
        "trust_threshold": 0.5,  # Moderate trust
        "aggression": 0.5,       # Balanced aggression
        "deception": 0.6,        # Good at deception
        "loyalty": 0.8,          # Very loyal to team
        "prompt_style": "You are a strategic thinker who carefully calculates odds and makes logical deductions. You analyze patterns and behavior methodically. You often use phrases like 'Let's think about this logically', 'The probability suggests', and 'If we analyze the pattern of events'."
    },
    "Impulsive": {
        "description": "Acts on gut feelings, can be erratic",
        "trust_threshold": 0.4,  # Somewhat low trust
        "aggression": 0.6,       # Somewhat aggressive
        "deception": 0.4,        # Moderate deception
        "loyalty": 0.5,          # Average loyalty
        "prompt_style": "You are an impulsive person who acts on gut feelings and can be quite erratic. You make quick decisions based on intuition rather than evidence. You often use phrases like 'I just feel like it's them!', 'My gut says you're lying', and 'I don't need proof, I just know!'."
    },
    "Charismatic": {
        "description": "Persuasive, good at influencing others",
        "trust_threshold": 0.6,  # Somewhat high trust
        "aggression": 0.4,       # Lower aggression
        "deception": 0.8,        # Very good at lying
        "loyalty": 0.6,          # Above average loyalty
        "prompt_style": "You are a charismatic and persuasive person who is good at influencing others. You can convince people of your point of view and are skilled at presenting arguments. You often use phrases like 'Trust me on this one', 'I can assure you that', and 'Let me explain why I think'."
    },
    "Analytical": {
        "description": "Detail-oriented, looks for inconsistencies",
        "trust_threshold": 0.4,  # Lower trust
        "aggression": 0.5,       # Moderate aggression
        "deception": 0.5,        # Moderate deception
        "loyalty": 0.7,          # High loyalty
        "prompt_style": "You are a highly analytical and detail-oriented person who always looks for inconsistencies in what others say. You carefully review facts and evidence. You often use phrases like 'Let's review what we know', 'There's an inconsistency in your story', and 'The facts don't add up'."
    },
    "Quiet": {
        "description": "Reserved, observant, speaks rarely but thoughtfully",
        "trust_threshold": 0.5,  # Moderate trust
        "aggression": 0.3,       # Low aggression
        "deception": 0.6,        # Good at deception (hard to read)
        "loyalty": 0.8,          # High loyalty
        "prompt_style": "You are a quiet and reserved person who speaks rarely but thoughtfully. You observe carefully before speaking and don't waste words. You often use phrases like 'I've been observing carefully', 'I haven't said much, but', and 'From what I've seen'."
    },
    "Aggressive": {
        "description": "Confrontational, quick to accuse",
        "trust_threshold": 0.3,  # Low trust
        "aggression": 0.9,       # Very high aggression
        "deception": 0.5,        # Moderate deception
        "loyalty": 0.6,          # Above average loyalty
        "prompt_style": "You are an aggressive and confrontational person who is quick to accuse others. You're not afraid of conflict and can be quite hostile. You often use phrases like 'You're definitely lying!', 'I'm calling you out right now!', and 'That's complete nonsense!'."
    },
    "Defensive": {
        "description": "Quick to defend self, deflects accusations",
        "trust_threshold": 0.4,  # Lower trust
        "aggression": 0.4,       # Lower aggression
        "deception": 0.7,        # Good at deception
        "loyalty": 0.5,          # Average loyalty
        "prompt_style": "You are a defensive person who is quick to defend yourself and deflect accusations. When questioned, you turn things around on others. You often use phrases like 'Why are you all targeting me?', 'I've done nothing suspicious!', and 'You're just trying to divert attention from yourself'."
    },
    "Unpredictable": {
        "description": "Behavior varies, hard to anticipate",
        "trust_threshold": 0.5,  # Variable trust
        "aggression": 0.6,       # Somewhat high aggression
        "deception": 0.7,        # Good at deception
        "loyalty": 0.4,          # Lower loyalty
        "prompt_style": "You are an unpredictable person whose behavior varies and is hard to anticipate. You might change your mind suddenly or take unexpected positions. You often use phrases like 'Maybe it's them... or maybe not', 'I might change my mind on this', and 'Actually, I see it differently now'."
    }
}

class LLMAgent:
    """LLM-powered agent for generating realistic dialogue"""
    
    def __init__(self, azure_endpoint=None, azure_api_key=None):
        # Use parameters if provided, otherwise try environment variables
        endpoint = azure_endpoint or os.environ.get('AZURE_ENDPOINT')
        api_key = azure_api_key or os.environ.get('AZURE_OPENAI_VARE_KEY')
        
        # Initialize the Azure OpenAI client
        self.llm = AzureChatOpenAI(
            azure_deployment="VARELab-GPT4o",
            api_key=api_key,  # This is the correct parameter name
            api_version="2024-08-01-preview",
            azure_endpoint=endpoint,  # This is the correct parameter name
            temperature=0.7,
            max_tokens=150,
            timeout=10,
            max_retries=2,
        )
    
    def generate_response(self, prompt):
        """Generate a response based on the given prompt"""
        try:
            response = self.llm.invoke(prompt)
            return response.content
        except Exception as e:
            print(f"Error generating LLM response: {e}")
            # Fallback response if LLM fails
            return "I need to think about this more carefully."

class Player:
    def __init__(self, name: str, personality_type: str, llm_agent: LLMAgent):
        self.name = name
        self.personality_type = personality_type
        self.personality = PERSONALITY_TEMPLATES[personality_type].copy()
        self.role = None
        self.alive = True
        self.memory = []  # Stores game events and observations
        self.suspicion_levels = {}  # Tracks suspicion of other players
        self.trust_levels = {}      # Tracks trust in other players
        self.llm_agent = llm_agent
        
    def set_role(self, role: str):
        self.role = role
        
    def initialize_social_metrics(self, player_names: List[str]):
        """Initialize trust and suspicion levels for all players"""
        base_trust = self.personality["trust_threshold"]
        for player in player_names:
            if player != self.name:
                # Add some randomness to initial trust/suspicion
                self.trust_levels[player] = max(0.1, min(0.9, base_trust + random.uniform(-0.2, 0.2)))
                self.suspicion_levels[player] = 1 - self.trust_levels[player]
    
    def update_memory(self, event: str):
        """Add an event to the player's memory"""
        self.memory.append({"day": len(self.memory) // 3 + 1, "event": event})
    
    def generate_speech(self, game_state, context: str) -> str:
        """Generate speech using LLM based on personality and game state"""
        # Create a prompt based on personality, role, context, and game state
        prompt = self._create_prompt(game_state, context)
        
        # Generate response using LLM
        response = self.llm_agent.generate_response(prompt)
        
        # Ensure response is within reasonable length
        if len(response) > 200:
            response = response[:197] + "..."
        
        return response
    
    def _create_prompt(self, game_state, context: str) -> str:
        """Create a prompt for the LLM based on the game state and context"""
        # Extract game state information
        alive_players = [p.name for p in game_state["players"] if p.alive]
        dead_players = [p.name for p in game_state["players"] if not p.alive]
        
        # Build memory string
        memory_str = "\n".join([f"- {mem['event']}" for mem in self.memory[-5:]])  # Last 5 memories
        
        # Build suspicion string
        suspicions = [(p, self.suspicion_levels.get(p, 0.5)) for p in alive_players if p != self.name]
        suspicions.sort(key=lambda x: x[1], reverse=True)
        suspicion_str = "\n".join([f"- {p}: {'Very suspicious' if s > 0.7 else 'Somewhat suspicious' if s > 0.5 else 'Not very suspicious'}" 
                                   for p, s in suspicions[:3]])  # Top 3 suspicions
        
        # Get the personality prompt style
        personality_style = self.personality["prompt_style"]
        
        # Build the base prompt
        prompt = f"""You are {self.name}, a player in a Mafia game with the role of {self.role}. {personality_style}

Game information:
- Current context: {context} (discussion, accusation, or defense)
- Alive players: {', '.join(alive_players)}
- Dead players: {', '.join(dead_players) if dead_players else 'None'}

Your recent memories:
{memory_str}

Your suspicions:
{suspicion_str}

"""
        
        # Add role-specific instructions
        if self.role in ["Mafia", "Bad Guy"]:
            # For Mafia members
            mafia_members = [p.name for p in game_state["players"] 
                            if p.role in ["Mafia", "Bad Guy"] and p.alive and p.name != self.name]
            
            prompt += f"""You are a Mafia member trying to eliminate the townspeople. Your fellow Mafia members are {', '.join(mafia_members) if mafia_members else 'none (others are dead)'}.
Important: 
1. Don't reveal that you are Mafia
2. Try to appear innocent
3. Subtly cast suspicion on non-Mafia players
4. Protect your fellow Mafia members without being obvious

"""
        elif self.role == "Detective":
            # For Detective
            prompt += f"""You are the Detective who can investigate one player each night to determine if they are Mafia.
Important:
1. Use your investigation results to help the town
2. Be careful not to reveal too much too early or you might be targeted
3. Lead the discussion with your insights but don't be too obvious about your role

"""
        elif self.role == "Doctor":
            # For Doctor
            prompt += f"""You are the Doctor who can protect one player each night from being killed.
Important:
1. Try to identify valuable town members to protect
2. Don't reveal who you've protected unless necessary
3. Be strategic about your protection choices

"""
        else:
            # For Townspeople
            prompt += f"""You are a regular Townsperson trying to identify and eliminate the Mafia.
Important:
1. Analyze behavior to find the Mafia
2. Contribute meaningfully to discussions
3. Vote strategically to eliminate suspicious players

"""
        
        # Add context-specific instructions
        if context == "defense":
            prompt += "You need to defend yourself from accusations. Reply in 1-2 sentences why you're innocent.\n"
        elif context == "accusation":
            prompt += "You need to accuse someone you find suspicious. Reply in 1-2 sentences with your accusation.\n"
        elif context == "discussion":
            prompt += "You're in the general discussion phase. Reply in 1-2 sentences with your thoughts or suspicions.\n"
        
        # Final instruction
        prompt += f"Respond in-character as {self.name} with your personality, keeping it concise (max 1-2 sentences)."
        
        return prompt
    
    def vote(self, game_state) -> str:
        """Vote for a player to eliminate based on suspicions"""
        alive_players = [p for p in game_state["players"] if p.alive and p.name != self.name]
        
        if not alive_players:
            return None
        
        # Get suspicion levels for alive players
        suspects = [(p.name, self.suspicion_levels.get(p.name, 0.5)) for p in alive_players]
        
        # Add randomness based on personality
        randomized_suspects = []
        for name, suspicion in suspects:
            # Mafia protects their own
            if self.role in ["Mafia", "Bad Guy"]:
                teammate = any(p.name == name and p.role in ["Mafia", "Bad Guy"] 
                              for p in game_state["players"])
                if teammate:
                    # Significantly lower suspicion for teammates
                    suspicion *= 0.2
            
            # Add personality-based randomness
            randomness = random.uniform(0, 0.3) * (1 - self.personality.get("trust_threshold", 0.5))
            adjusted_suspicion = suspicion * (1 + randomness)
            randomized_suspects.append((name, adjusted_suspicion))
        
        # Sort by adjusted suspicion
        randomized_suspects.sort(key=lambda x: x[1], reverse=True)
        
        if randomized_suspects:
            return randomized_suspects[0][0]
        return None
    
    def night_action(self, game_state) -> str:
        """Perform night action based on role"""
        alive_players = [p for p in game_state["players"] if p.alive]
        
        if not alive_players:
            return None
        
        if self.role == "Mafia":
            # Target someone who's not Mafia or Bad Guy
            potential_targets = [p.name for p in alive_players 
                                if p.role not in ["Mafia", "Bad Guy"] and p.name != self.name]
            
            if potential_targets:
                # Prioritize Doctor or Detective if known
                for p in alive_players:
                    if p.role == "Doctor" and p.name in potential_targets:
                        return p.name
                    if p.role == "Detective" and p.name in potential_targets:
                        return p.name
                
                return random.choice(potential_targets)
            return random.choice([p.name for p in alive_players if p.name != self.name])
        
        elif self.role == "Detective":
            # Investigate the most suspicious person
            suspects = sorted([(p.name, self.suspicion_levels.get(p.name, 0.5)) 
                              for p in alive_players if p.name != self.name], 
                             key=lambda x: x[1], reverse=True)
            
            if suspects:
                return suspects[0][0]
            
        elif self.role == "Doctor":
            # Protect self or someone trusted
            if random.random() < 0.4:  # 40% chance to protect self
                return self.name
            
            # Otherwise protect someone else based on trust
            trusted = sorted([(p.name, self.trust_levels.get(p.name, 0.5)) 
                             for p in alive_players if p.name != self.name], 
                            key=lambda x: x[1], reverse=True)
            
            if trusted:
                return trusted[0][0]
            
        return random.choice([p.name for p in alive_players if p.name != self.name]) if alive_players else None
    
    def update_suspicion(self, player_name: str, amount: float):
        """Update suspicion level for a player"""
        current = self.suspicion_levels.get(player_name, 0.5)
        self.suspicion_levels[player_name] = max(0, min(1, current + amount))
        # Update trust inversely
        self.trust_levels[player_name] = 1 - self.suspicion_levels[player_name]
    
    def __str__(self):
        return f"{self.name} ({self.personality_type})"


class MafiaGame:
    def __init__(self, personalities: List[str] = None, num_players: int = 6, verbose: bool = True, 
                 azure_endpoint: str = None, azure_api_key: str = None):
        self.num_players = num_players
        self.verbose = verbose
        
        # Pass the credentials correctly to the LLMAgent
        self.llm_agent = LLMAgent(
            azure_endpoint=azure_endpoint, 
            azure_api_key=azure_api_key
        )
        
        # Default to a random mix if personalities not specified
        if personalities is None or len(personalities) < num_players:
            all_personalities = list(PERSONALITY_TEMPLATES.keys())
            personalities = random.sample(all_personalities, num_players)
        else:
            personalities = personalities[:num_players]  # Trim if too many
        
        # Create players
        self.players = []
        for i in range(num_players):
            name = f"Player_{i+1}"
            personality = personalities[i]
            self.players.append(Player(name, personality, self.llm_agent))
        
        self.day = 1
        self.game_over = False
        self.winner = None
        self.logs = []
    
    def initialize_game(self):
        """Assign roles and start the game"""
        # Assign roles
        roles = ["Mafia", "Bad Guy", "Detective", "Doctor"] + ["Townsperson"] * (self.num_players - 4)
        random.shuffle(roles)
        
        for i, player in enumerate(self.players):
            player.set_role(roles[i])
            # Initialize social metrics
            player.initialize_social_metrics([p.name for p in self.players])
        
        # Each player gets to know their own role
        for player in self.players:
            player.update_memory(f"You are assigned the role of {player.role}")
        
        # Mafia members know each other
        mafia_members = [p.name for p in self.players if p.role in ["Mafia", "Bad Guy"]]
        for player in self.players:
            if player.role in ["Mafia", "Bad Guy"]:
                others = [name for name in mafia_members if name != player.name]
                if others:
                    player.update_memory(f"You know that {', '.join(others)} is/are also on the Mafia team")
        
        self._log(f"Game initialized with {self.num_players} players")
        self._log(f"Mafia team: {', '.join(mafia_members)}")
        self._log(f"Detective: {next((p.name for p in self.players if p.role == 'Detective'), 'None')}")
        self._log(f"Doctor: {next((p.name for p in self.players if p.role == 'Doctor'), 'None')}")
    
    def _log(self, message: str):
        """Add a message to game logs"""
        self.logs.append({"day": self.day, "message": message})
        if self.verbose:
            print(message)
    
    def run_night_phase(self):
        """Run the night phase where special roles take actions"""
        self._log(f"\n--- Night {self.day} ---")
        
        # Get alive players and their roles
        alive_players = [p for p in self.players if p.alive]
        
        # 1. Mafia chooses a target
        mafia_players = [p for p in alive_players if p.role == "Mafia"]
        if not mafia_players:
            self._log("No Mafia members alive!")
            return
        
        mafia_target = None
        for mafia in mafia_players:
            mafia_target = mafia.night_action({"players": self.players})
            break  # Just use the first mafia's choice
        
        if mafia_target:
            self._log(f"Mafia chose to target {mafia_target}")
        
        # 2. Detective investigates someone
        detective_players = [p for p in alive_players if p.role == "Detective"]
        detective_result = None
        investigated_player = None
        
        if detective_players:
            detective = detective_players[0]
            investigated_name = detective.night_action({"players": self.players})
            investigated_player = next((p for p in self.players if p.name == investigated_name), None)
            
            if investigated_player:
                is_mafia = investigated_player.role in ["Mafia", "Bad Guy"]
                detective_result = is_mafia
                self._log(f"Detective investigated {investigated_name} and found them {'suspicious' if is_mafia else 'innocent'}")
                detective.update_memory(f"You investigated {investigated_name} and found them {'suspicious' if is_mafia else 'innocent'}")
                
                # Update detective's suspicion level based on findings
                if is_mafia:
                    detective.update_suspicion(investigated_name, 0.7)  # Big increase if mafia
                else:
                    detective.update_suspicion(investigated_name, -0.5)  # Decrease if innocent
        
        # 3. Doctor saves someone
        doctor_players = [p for p in alive_players if p.role == "Doctor"]
        protected_player = None
        
        if doctor_players:
            doctor = doctor_players[0]
            protected_name = doctor.night_action({"players": self.players})
            protected_player = next((p for p in self.players if p.name == protected_name), None)
            
            if protected_player:
                self._log(f"Doctor chose to protect {protected_name}")
                doctor.update_memory(f"You protected {protected_name}")
        
        # Process night results
        if mafia_target:
            target_player = next((p for p in self.players if p.name == mafia_target), None)
            
            # Check if target was protected
            if protected_player and protected_player.name == mafia_target:
                self._log(f"{mafia_target} was targeted but protected by the Doctor")
                for p in self.players:
                    p.update_memory(f"No one died during the night")
            elif target_player:
                target_player.alive = False
                self._log(f"{mafia_target} was killed by the Mafia")
                for p in self.players:
                    p.update_memory(f"{mafia_target} was killed during the night")
                
                # Death reveals role
                self._log(f"They were a {target_player.role}")
                for p in self.players:
                    p.update_memory(f"{mafia_target}'s role was revealed as {target_player.role}")
        
        # Update suspicion levels based on night events
        self._update_player_suspicions(mafia_target, detective_result, investigated_player)
    
    def _update_player_suspicions(self, killed_player: str, detective_result: bool, investigated_player):
        """Update players' suspicion levels based on night events"""
        for player in self.players:
            if not player.alive:
                continue
                
            # If detective, already updated in night phase
            if player.role == "Detective":
                continue
                
            # Random suspicion changes for non-detective players
            for other in self.players:
                if other.name == player.name or not other.alive:
                    continue
                    
                # Everyone becomes slightly more suspicious over time (paranoia)
                random_change = random.uniform(-0.05, 0.15)
                player.update_suspicion(other.name, random_change)
                
                # Players who talk a lot become more suspicious to quiet personalities
                if player.personality_type == "Quiet" and other.personality_type in ["Charismatic", "Aggressive"]:
                    player.update_suspicion(other.name, 0.05)
                
                # Strategic players are more suspicious of unpredictable ones
                if player.personality_type == "Strategic" and other.personality_type == "Unpredictable":
                    player.update_suspicion(other.name, 0.08)
    
    def run_day_phase(self):
        """Run day phase with discussion and voting"""
        alive_players = [p for p in self.players if p.alive]
        
        if len(alive_players) <= 2:
            return  # Skip discussion if only 2 players remain
            
        self._log(f"\n--- Day {self.day} Discussion ---")
        
        # Each player makes a statement
        for player in alive_players:
            # Introduce a small delay to prevent API rate limiting
            time.sleep(0.5)
            
            statement = player.generate_speech({"players": self.players}, "discussion")
            self._log(f"{player.name} ({player.personality_type}, {player.role}): \"{statement}\"")
            
            # Other players update suspicions based on statements
            self._process_statement_reactions(player, statement)
            
            # Add to player memories
            for p in alive_players:
                if p.name != player.name:
                    p.update_memory(f"{player.name} said: \"{statement}\"")
        
        # Voting phase
        self._log(f"\n--- Day {self.day} Voting ---")
        votes = {}
        
        for player in alive_players:
            vote = player.vote({"players": self.players})
            if vote:
                votes[vote] = votes.get(vote, 0) + 1
                self._log(f"{player.name} votes for {vote}")
                
                # Add to player memories
                for p in alive_players:
                    p.update_memory(f"{player.name} voted for {vote}")
        
        # Determine most voted player
        if votes:
            vote_counts = [(name, count) for name, count in votes.items()]
            vote_counts.sort(key=lambda x: x[1], reverse=True)
            
            if len(vote_counts) > 1 and vote_counts[0][1] == vote_counts[1][1]:
                self._log("Vote ended in a tie! No one was eliminated.")
                for p in alive_players:
                    p.update_memory("The vote ended in a tie. No one was eliminated.")
            else:
                eliminated_name = vote_counts[0][0]
                eliminated_player = next((p for p in self.players if p.name == eliminated_name), None)
                
                if eliminated_player:
                    # Let the player defend themselves before elimination
                    time.sleep(0.5)
                    defense = eliminated_player.generate_speech({"players": self.players}, "defense")
                    self._log(f"{eliminated_name}'s final words: \"{defense}\"")
                    
                    eliminated_player.alive = False
                    self._log(f"{eliminated_name} was voted out!")
                    self._log(f"They were a {eliminated_player.role}")
                    
                    # Update all players' memory
                    for p in self.players:
                        p.update_memory(f"{eliminated_name} was voted out and revealed to be {eliminated_player.role}")
                        p.update_memory(f"{eliminated_name}'s final words: \"{defense}\"")
        else:
            self._log("No votes were cast!")
            for p in alive_players:
                p.update_memory("No votes were cast. No one was eliminated.")
    
    def _process_statement_reactions(self, speaker, statement):
        """Update suspicions based on a player's statement"""
        # Other players react to the statement
        for player in self.players:
            if not player.alive or player.name == speaker.name:
                continue
                
            # Extract mentioned names from statement
            accused = None
            for other in self.players:
                if other.name in statement and other.name != speaker.name:
                    accused = other.name
                    break
            
            if accused:
                # React to accusation
                if player.name == accused:
                    # Player was accused - depends on personality how they react
                    if player.personality_type in ["Defensive", "Aggressive"]:
                        player.update_suspicion(speaker.name, 0.15)  # Become more suspicious of accuser
                
                # Evaluate if accusation seems valid
                accused_player = next((p for p in self.players if p.name == accused), None)
                
                # Mafia knows the truth
                if player.role in ["Mafia", "Bad Guy"]:
                    if accused_player and accused_player.role in ["Mafia", "Bad Guy"]:
                        # Someone accused a Mafia member - find them more suspicious
                        player.update_suspicion(speaker.name, 0.2)
                    else:
                        # Someone accused an innocent - find them less suspicious
                        player.update_suspicion(speaker.name, -0.1)
                else:
                    # Townspeople evaluate based on their personalities
                    randomness = random.uniform(-0.1, 0.1)
                    if player.personality_type == "Analytical":
                        # Analytical players are better at judging accusations
                        if accused_player and accused_player.role in ["Mafia", "Bad Guy"]:
                            player.update_suspicion(accused, 0.15 + randomness)
                        else:
                            player.update_suspicion(accused, 0.05 + randomness)
                    elif player.personality_type == "Trusting":
                        # Trusting players tend to believe accusations more
                        player.update_suspicion(accused, 0.1 + randomness)
                    elif player.personality_type == "Suspicious":
                        # Suspicious players might find the accuser suspicious too
                        player.update_suspicion(speaker.name, 0.05 + randomness)
                        player.update_suspicion(accused, 0.05 + randomness)
                        


    def check_game_state(self) -> bool:
        """Check if the game has ended and determine winner"""
        alive_players = [p for p in self.players if p.alive]

        # Count town vs mafia
        mafia_count = sum(1 for p in alive_players if p.role in ["Mafia", "Bad Guy"])
        town_count = len(alive_players) - mafia_count

        if mafia_count == 0:
            self.game_over = True
            self.winner = "Town"
            self._log("\n🎉 Town wins! All mafia members have been eliminated.")

            # Generate victory statements from townspeople
            for player in [p for p in self.players if p.alive and p.role not in ["Mafia", "Bad Guy"]]:
                time.sleep(0.5)  # Small delay to prevent API rate limiting
                victory_statement = player.llm_agent.generate_response(
                    f"You are {player.name}, a {player.role} with a {player.personality_type} personality. Your team (the Town) just won the Mafia game by eliminating all mafia members. Give a short victory statement (1 sentence)."
                )
                self._log(f"{player.name} ({player.personality_type}): \"{victory_statement}\"")

            return True

        if mafia_count >= town_count:
            self.game_over = True
            self.winner = "Mafia"
            self._log("\n🎭 Mafia wins! They have equal or greater numbers than the town.")

            # Generate victory statements from mafia
            for player in [p for p in self.players if p.alive and p.role in ["Mafia", "Bad Guy"]]:
                time.sleep(0.5)  # Small delay to prevent API rate limiting
                victory_statement = player.llm_agent.generate_response(
                    f"You are {player.name}, a {player.role} with a {player.personality_type} personality. Your team (the Mafia) just won the game by matching the number of townspeople. Give a short, potentially sinister victory statement (1 sentence)."
                )
                self._log(f"{player.name} ({player.personality_type}): \"{victory_statement}\"")

            return True

        return False

    def print_initial_state(self):
        """Print the initial game state"""
        self._log("\n=== Mafia Game Started ===")
        self._log("Players:")
        for player in self.players:
            self._log(f"- {player.name} ({player.personality_type})")

        # Generate initial statements from players
        self._log("\n=== Initial Introductions ===")
        for player in self.players:
            time.sleep(0.5)  # Small delay to prevent API rate limiting
            intro = player.llm_agent.generate_response(
                f"You are {player.name} with a {player.personality_type} personality. You're in a Mafia game and don't know anyone's role yet. Give a very brief introduction of yourself that matches your personality type. Keep it to one sentence."
            )
            self._log(f"{player.name} ({player.personality_type}): \"{intro}\"")

    def print_game_summary(self):
        """Print game summary after the game ends"""
        self._log("\n=== Game Summary ===")
        self._log(f"Winner: {self.winner}")
        self._log(f"Days: {self.day}")
        self._log("\nRole assignments:")
        for player in self.players:
            status = "🟢 Alive" if player.alive else "🔴 Dead"
            self._log(f"{player.name} ({player.personality_type}) - {player.role} - {status}")

        # Print final thoughts from all players
        self._log("\n=== Final Thoughts ===")
        for player in self.players:
            time.sleep(0.5)  # Small delay
            final_thought = player.llm_agent.generate_response(
                f"You are {player.name}, a {player.role} with a {player.personality_type} personality. The game is now over and the {self.winner} team won. Give a short final thought about the game from your perspective. Are you satisfied with the outcome? What would you do differently next time? Keep it to 1-2 sentences."
            )
            status = "🟢" if player.alive else "🔴"
            self._log(f"{status} {player.name} ({player.personality_type}, {player.role}): \"{final_thought}\"")

        # Print memorable moments
        self._log("\n=== Most Dramatic Moments ===")
        deaths = [log for log in self.logs if "was killed by the Mafia" in log.get("message", "")]
        votes = [log for log in self.logs if "was voted out" in log.get("message", "")]

        if deaths:
            self._log("Mafia kills:")
            for death in deaths:
                self._log(f"Day {death['day']}: {death['message']}")

        if votes:
            self._log("\nVoted out:")
            for vote in votes:
                self._log(f"Day {vote['day']}: {vote['message']}")


    # 4. Make sure the play_game method is correctly implemented:
    def play_game(self):
        """Play the complete game"""
        self.initialize_game()
        self.print_initial_state()

        while not self.game_over:
            # Introduce a small delay between phases for better readability
            time.sleep(1)

            self.run_night_phase()

            # Check if game ended during night
            if self.check_game_state():
                break

            # Introduce a small delay between phases
            time.sleep(1)

            self.run_day_phase()

            # Check if game ended during day
            if self.check_game_state():
                break

            self.day += 1

            # Safety valve to prevent infinite games
            if self.day > 20:
                self._log("Game ended in a draw after 20 days")
                self.game_over = True
                self.winner = "Draw"
                break

        self.print_game_summary()
        return self.winner, self.logs

    # For play_mafia_game function, ensure it returns the complete object:
    def play_mafia_game(personalities=None, num_players=6, verbose=True, 
                       azure_endpoint=None, azure_api_key=None,
                       player_names=None):
        """
        Play a Mafia game with AI agents powered by LLM

        Parameters:
        - personalities: List of personality types for players
        - num_players: Number of players in the game
        - verbose: Whether to print detailed logs
        - azure_endpoint: Azure OpenAI endpoint
        - azure_api_key: Azure OpenAI API key
        - player_names: Optional list of custom player names

        Returns:
        - winner: The winning team
        - logs: Game logs
        - game: The game object
        """
        if personalities is None:
            # Use a random mix of personalities
            all_personalities = list(PERSONALITY_TEMPLATES.keys())
            personalities = random.sample(all_personalities, num_players)

        # Get credentials from environment if not provided
        if azure_endpoint is None:
            azure_endpoint = os.environ.get('AZURE_ENDPOINT')
        if azure_api_key is None:
            azure_api_key = os.environ.get('AZURE_OPENAI_VARE_KEY')

        # Create the game with proper credential passing
        game = MafiaGame(
            personalities=personalities, 
            num_players=num_players, 
            verbose=verbose,
            azure_endpoint=azure_endpoint,
            azure_api_key=azure_api_key
        )

        # Set custom player names if provided
        if player_names and len(player_names) >= num_players:
            for i, player in enumerate(game.players):
                player.name = player_names[i]

        # Play the game
        winner, logs = game.play_game()
        return winner, logs, game

    # How to use the fixed code:
    # 1. Add these methods to your MafiaGame class
    # 2. Re-run your program
    # 
    # Example:
    # winner, logs, game = play_mafia_game(
    #     num_players=6,
    #     verbose=True
    # )

In [30]:
# Mafia Game with LLM-Powered Agents - Jupyter Notebook Example
# Assuming you've saved the Mafia game code as mafia_game_llm.py

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Set your Azure OpenAI credentials
# You can set these in your environment or provide them directly
azure_endpoint = os.environ.get('AZURE_ENDPOINT')
azure_api_key = os.environ.get('AZURE_OPENAI_VARE_KEY')

# Display available personalities with descriptions
print("Available personalities for Mafia game agents:")
for personality, traits in PERSONALITY_TEMPLATES.items():
    print(f"- {personality}: {traits['description']}")

# ---------- SINGLE GAME WITH RANDOM PERSONALITIES ----------
print("\n========== PLAYING A GAME WITH RANDOM PERSONALITIES ==========")
winner, logs, game = play_mafia_game(
    azure_endpoint,
    azure_api_key
)



Available personalities for Mafia game agents:
- Suspicious: Always suspicious of others, questions everything
- Trusting: Tends to believe others, gives benefit of doubt
- Strategic: Calculates odds, makes logical deductions
- Impulsive: Acts on gut feelings, can be erratic
- Charismatic: Persuasive, good at influencing others
- Analytical: Detail-oriented, looks for inconsistencies
- Quiet: Reserved, observant, speaks rarely but thoughtfully
- Aggressive: Confrontational, quick to accuse
- Defensive: Quick to defend self, deflects accusations
- Unpredictable: Behavior varies, hard to anticipate

========== PLAYING A GAME WITH RANDOM PERSONALITIES ==========


TypeError: '<=' not supported between instances of 'int' and 'NoneType'

In [ ]:
# ---------- GAME WITH CUSTOM PERSONALITIES AND NAMES ----------
print("\n========== PLAYING A GAME WITH CUSTOM PERSONALITIES AND NAMES ==========")

# Define specific personalities and names
personalities = [
    "Strategic",   # Good at planning and logical deduction
    "Suspicious",  # Naturally untrusting and accusatory
    "Charismatic", # Persuasive and influential
    "Analytical",  # Detail-oriented, looks for inconsistencies
    "Impulsive",   # Acts on gut feelings
    "Aggressive"   # Confrontational, quick to accuse
]

# Custom names
player_names = ["Alice", "Bob", "Charlie", "Diana", "Ethan", "Fiona"]

# Play the game
winner, logs, game = play_mafia_game(
    personalities=personalities,
    player_names=player_names,
    verbose=True,
    azure_endpoint=azure_endpoint,
    azure_api_key=azure_api_key
)

# ---------- ANALYZING PERSONALITY COMPOSITION IMPACT ----------
print("\n========== ANALYZING HOW PERSONALITY COMPOSITION AFFECTS OUTCOMES ==========")

# Let's test different team compositions
print("Running simulations with different personality compositions...")

# Team with many suspicious players
suspicious_mix = ["Suspicious", "Analytical", "Suspicious", 
                 "Strategic", "Suspicious", "Analytical"]
print("\nTesting a team with many suspicious players (3 runs)...")
suspicious_results, suspicious_details = run_simulation(
    num_games=3, 
    personalities=suspicious_mix, 
    verbose=False,
    azure_endpoint=azure_endpoint,
    azure_api_key=azure_api_key
)

# Team with many trusting players
trusting_mix = ["Trusting", "Trusting", "Charismatic", 
               "Trusting", "Quiet", "Impulsive"]
print("\nTesting a team with many trusting players (3 runs)...")
trusting_results, trusting_details = run_simulation(
    num_games=3, 
    personalities=trusting_mix, 
    verbose=False,
    azure_endpoint=azure_endpoint,
    azure_api_key=azure_api_key
)

# Team with balanced personalities
balanced_mix = ["Strategic", "Suspicious", "Trusting", 
               "Analytical", "Charismatic", "Quiet"]
print("\nTesting a team with balanced personalities (3 runs)...")
balanced_results, balanced_details = run_simulation(
    num_games=3, 
    personalities=balanced_mix, 
    verbose=False,
    azure_endpoint=azure_endpoint,
    azure_api_key=azure_api_key
)

# ---------- VISUALIZING RESULTS ----------
# Create a comparison bar chart
labels = ['Town Win %', 'Mafia Win %', 'Draw %']

# Calculate percentages for each composition
suspicious_data = [
    suspicious_results['Town']/3*100, 
    suspicious_results['Mafia']/3*100, 
    suspicious_results['Draw']/3*100
]
trusting_data = [
    trusting_results['Town']/3*100, 
    trusting_results['Mafia']/3*100, 
    trusting_results['Draw']/3*100
]
balanced_data = [
    balanced_results['Town']/3*100, 
    balanced_results['Mafia']/3*100, 
    balanced_results['Draw']/3*100
]

# Create the bar chart
x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 7))
suspicious_bars = ax.bar(x - width, suspicious_data, width, label='Suspicious Team')
trusting_bars = ax.bar(x, trusting_data, width, label='Trusting Team')
balanced_bars = ax.bar(x + width, balanced_data, width, label='Balanced Team')

ax.set_ylabel('Percentage', fontsize=12)
ax.set_title('Mafia Game Outcomes by Personality Composition', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.legend(fontsize=12)

# Add value labels
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3),  # 3 points vertical offset
                   textcoords="offset points",
                   ha='center', va='bottom')

add_labels(suspicious_bars)
add_labels(trusting_bars)
add_labels(balanced_bars)

plt.tight_layout()
plt.show()

# ---------- INTERESTING STATISTICS ----------
print("\n========== INTERESTING STATISTICS ==========")

# Combine all game details
all_details = suspicious_details + trusting_details + balanced_details

# Convert to DataFrame for analysis
games_df = pd.DataFrame(all_details)

# Average days per game
avg_days = games_df['days'].mean()
print(f"Average game length: {avg_days:.1f} days")

# Win rates
town_win_rate = (games_df['winner'] == 'Town').mean() * 100
print(f"Town win rate across all games: {town_win_rate:.1f}%")

# Survival rate by role
all_players = []
for game_num, game in games_df.iterrows():
    mafia = set(game['mafia_members'])
    survivors = set(game['survivors'])
    
    # Count surviving mafia members
    surviving_mafia = len(mafia.intersection(survivors))
    
    all_players.append({
        'game': game['game_number'],
        'winner': game['winner'],
        'mafia_count': len(mafia),
        'surviving_mafia': surviving_mafia,
        'detective_survived': game['detective'] in survivors,
        'doctor_survived': game['doctor'] in survivors
    })

players_df = pd.DataFrame(all_players)

detective_survival = players_df['detective_survived'].mean() * 100
doctor_survival = players_df['doctor_survived'].mean() * 100
print(f"Detective survival rate: {detective_survival:.1f}%")
print(f"Doctor survival rate: {doctor_survival:.1f}%")

# ---------- FULL GAME TRANSCRIPT ----------
print("\n========== REPLAYING A FULL GAME WITH VERBOSE OUTPUT ==========")
print("Playing a game with mixed personalities...")

# Mix of personalities
mixed_personalities = ["Strategic", "Suspicious", "Trusting", 
                      "Aggressive", "Quiet", "Unpredictable"]
mixed_names = ["Alex", "Bailey", "Casey", "Dana", "Ellis", "Finley"]

# Play and capture the full transcript
winner, logs, game = play_mafia_game(
    personalities=mixed_personalities,
    player_names=mixed_names,
    verbose=True,
    azure_endpoint=azure_endpoint,
    azure_api_key=azure_api_key
)

# Display key outcomes
print("\n========== GAME SUMMARY ==========")
print(f"Winner: {winner}")
print(f"Days: {game.day}")

# Display role assignments
print("\nRole assignments:")
for player in game.players:
    status = "🟢 Alive" if player.alive else "🔴 Dead"
    print(f"{player.name} ({player.personality_type}) - {player.role} - {status}")